[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/agent-sdk/01-primeros-pasos.ipynb)

# Primeros Pasos con el Agent SDK

Este notebook implementa un agente básico con herramientas y demuestra el bucle agéntico completo.

In [ ]:
import anthropic
import json
from concurrent.futures import ThreadPoolExecutor

client = anthropic.Anthropic()
print("Cliente Anthropic inicializado")

## 1. Definir herramientas y ejecutor

In [ ]:
HERRAMIENTAS = [
    {
        "name": "buscar_web",
        "description": "Busca información actualizada en internet",
        "input_schema": {
            "type": "object",
            "properties": {
                "consulta": {"type": "string", "description": "Términos de búsqueda"}
            },
            "required": ["consulta"]
        }
    },
    {
        "name": "calcular",
        "description": "Realiza cálculos matemáticos",
        "input_schema": {
            "type": "object",
            "properties": {
                "expresion": {"type": "string", "description": "Expresión matemática, ej: 2+2*3"}
            },
            "required": ["expresion"]
        }
    },
    {
        "name": "obtener_tipo_cambio",
        "description": "Obtiene el tipo de cambio entre dos monedas",
        "input_schema": {
            "type": "object",
            "properties": {
                "de": {"type": "string", "description": "Moneda origen, ej: EUR"},
                "a": {"type": "string", "description": "Moneda destino, ej: USD"}
            },
            "required": ["de", "a"]
        }
    }
]

# Base de datos simulada de tipos de cambio
TIPOS_CAMBIO = {
    "EUR_USD": 1.085,
    "EUR_GBP": 0.857,
    "USD_EUR": 0.922,
    "USD_GBP": 0.790,
    "GBP_EUR": 1.167,
    "GBP_USD": 1.265
}

def ejecutar_herramienta(nombre: str, parametros: dict) -> str:
    if nombre == "buscar_web":
        consulta = parametros['consulta']
        # Simulación de búsqueda
        resultados = {
            "iva españa": "El IVA general en España es del 21%. Existe IVA reducido (10%) para alimentos, transportes y hostelería, y superreducido (4%) para productos de primera necesidad.",
            "default": f"Resultados de búsqueda para '{consulta}': Información actualizada encontrada sobre el tema."
        }
        for k, v in resultados.items():
            if k.lower() in consulta.lower():
                return v
        return resultados["default"]
    
    elif nombre == "calcular":
        try:
            resultado = eval(parametros["expresion"], {"__builtins__": {}})
            return f"{parametros['expresion']} = {resultado}"
        except Exception as e:
            return f"Error en el cálculo: {e}"
    
    elif nombre == "obtener_tipo_cambio":
        clave = f"{parametros['de'].upper()}_{parametros['a'].upper()}"
        if clave in TIPOS_CAMBIO:
            return f"1 {parametros['de'].upper()} = {TIPOS_CAMBIO[clave]} {parametros['a'].upper()}"
        return f"Tipo de cambio {clave} no disponible"
    
    return "Herramienta no reconocida"

print("Herramientas definidas:", [h["name"] for h in HERRAMIENTAS])

## 2. Bucle agéntico básico

In [ ]:
def agente(objetivo: str, max_iteraciones: int = 10, verbose: bool = True) -> str:
    mensajes = [{"role": "user", "content": objetivo}]
    
    if verbose:
        print(f"🎯 Objetivo: {objetivo}\n")

    for i in range(max_iteraciones):
        respuesta = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=HERRAMIENTAS,
            messages=mensajes
        )

        mensajes.append({"role": "assistant", "content": respuesta.content})

        if respuesta.stop_reason == "end_turn":
            for bloque in respuesta.content:
                if hasattr(bloque, "text"):
                    if verbose:
                        print(f"\n✅ Respuesta final ({i+1} iteraciones):")
                    return bloque.text
            return "Completado"

        if respuesta.stop_reason == "tool_use":
            resultados_herramientas = []
            for bloque in respuesta.content:
                if bloque.type == "tool_use":
                    if verbose:
                        print(f"  [{i+1}] 🔧 {bloque.name}({bloque.input})")
                    resultado = ejecutar_herramienta(bloque.name, bloque.input)
                    if verbose:
                        print(f"       → {resultado[:100]}")
                    resultados_herramientas.append({
                        "type": "tool_result",
                        "tool_use_id": bloque.id,
                        "content": resultado
                    })
            mensajes.append({"role": "user", "content": resultados_herramientas})

    return "Máximo de iteraciones alcanzado"


# Prueba 1: cálculo + búsqueda
resultado = agente("¿Cuánto es el 15% de 2.847€? Y busca qué es el IVA en España")
print(resultado)

In [ ]:
# Prueba 2: múltiples herramientas en secuencia
resultado2 = agente("Tengo 1500 EUR. ¿Cuántos USD son? ¿Y cuánto sería el 8% de esa cantidad en USD?")
print(resultado2)

## 3. Parallel tool use

Claude puede solicitar múltiples herramientas a la vez. Aquí las ejecutamos en paralelo.

In [ ]:
def agente_paralelo(objetivo: str, max_iteraciones: int = 10) -> str:
    mensajes = [{"role": "user", "content": objetivo}]
    print(f"🎯 Objetivo: {objetivo}\n")

    for i in range(max_iteraciones):
        respuesta = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=HERRAMIENTAS,
            messages=mensajes
        )
        mensajes.append({"role": "assistant", "content": respuesta.content})

        if respuesta.stop_reason == "end_turn":
            for bloque in respuesta.content:
                if hasattr(bloque, "text"):
                    return bloque.text
            return "Completado"

        if respuesta.stop_reason == "tool_use":
            bloques_tool = [b for b in respuesta.content if b.type == "tool_use"]
            
            if len(bloques_tool) > 1:
                print(f"  [{i+1}] ⚡ Ejecutando {len(bloques_tool)} herramientas en paralelo")
            
            resultados = []
            with ThreadPoolExecutor(max_workers=4) as executor:
                futuros = {
                    executor.submit(ejecutar_herramienta, b.name, b.input): b
                    for b in bloques_tool
                }
                for futuro, bloque in futuros.items():
                    resultado = futuro.result()
                    print(f"       🔧 {bloque.name} → {resultado[:80]}")
                    resultados.append({
                        "type": "tool_result",
                        "tool_use_id": bloque.id,
                        "content": str(resultado)
                    })
            mensajes.append({"role": "user", "content": resultados})

    return "Máximo de iteraciones"

# Consulta que puede aprovechar parallel tool use
resultado3 = agente_paralelo(
    "Obtén el tipo de cambio EUR/USD y EUR/GBP al mismo tiempo, "
    "luego calcula cuánto es 1000€ en cada moneda"
)
print("\n" + resultado3)

## 4. Manejo de errores en herramientas

In [ ]:
import time

def ejecutar_herramienta_segura(nombre: str, parametros: dict, timeout: float = 5.0) -> dict:
    """Ejecuta herramienta con timeout y manejo de errores."""
    inicio = time.time()
    try:
        resultado = ejecutar_herramienta(nombre, parametros)
        latencia = time.time() - inicio
        print(f"  ✓ {nombre} ({latencia:.2f}s)")
        return {"success": True, "resultado": resultado}
    except Exception as e:
        print(f"  ✗ {nombre} falló: {e}")
        return {"success": False, "error": f"{type(e).__name__}: {str(e)}"}

# Probar herramienta que falla
r1 = ejecutar_herramienta_segura("calcular", {"expresion": "1 / 0"})
print("Resultado:", r1)

r2 = ejecutar_herramienta_segura("calcular", {"expresion": "15 * 2847 / 100"})
print("Resultado:", r2)

## 5. System prompt para agentes

In [ ]:
SYSTEM_AGENTE = """Eres un agente de análisis financiero con acceso a herramientas.

COMPORTAMIENTO:
- Analiza cuidadosamente la tarea antes de actuar
- Usa herramientas solo cuando sean necesarias
- Verifica los resultados antes de concluir
- Si una herramienta falla, intenta un enfoque alternativo
- Sé conciso en respuestas intermedias, detallado en la respuesta final

REGLAS:
- No inventes tipos de cambio; usa la herramienta para obtenerlos
- Si no puedes completar la tarea, explica por qué
- Confirma resultados calculados antes de presentarlos
"""

def agente_financiero(objetivo: str) -> str:
    mensajes = [{"role": "user", "content": objetivo}]
    print(f"🎯 {objetivo}\n")

    for i in range(10):
        respuesta = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system=SYSTEM_AGENTE,
            tools=HERRAMIENTAS,
            messages=mensajes
        )
        mensajes.append({"role": "assistant", "content": respuesta.content})

        if respuesta.stop_reason == "end_turn":
            return next((b.text for b in respuesta.content if hasattr(b, "text")), "")

        if respuesta.stop_reason == "tool_use":
            resultados = []
            for bloque in respuesta.content:
                if bloque.type == "tool_use":
                    print(f"  [{i+1}] 🔧 {bloque.name}")
                    r = ejecutar_herramienta_segura(bloque.name, bloque.input)
                    contenido = r["resultado"] if r["success"] else f"ERROR: {r['error']}"
                    resultados.append({
                        "type": "tool_result",
                        "tool_use_id": bloque.id,
                        "content": contenido
                    })
            mensajes.append({"role": "user", "content": resultados})

    return "Máximo de iteraciones"

resultado = agente_financiero(
    "Analiza: tengo un presupuesto de 50.000 EUR. ¿Cuánto es en USD y GBP? "
    "Si invierto el 30% de cada cantidad, ¿cuánto tendré en cada moneda?"
)
print("\n" + resultado)